# odmlib v0.2.0: Permissive Loading Mode

ODM and Define-XML files encountered in practice often contain validation violations: missing
required attributes, invalid enumeration values, or wrong data types. Before v0.2.0, odmlib
rejected these files on load, which made it impossible to inspect or repair them
programmatically.

odmlib v0.2.0 introduces **permissive mode**, a context-managed validation control system that
selectively bypasses validation checks so that non-conformant documents can be loaded for
inspection and repair. Permissive mode uses Python's `contextvars` module, so it is
thread-safe and supports nesting.

This notebook covers:

1. **The problem** — what happens when strict mode encounters non-conformant content
2. **The `permissive()` context manager** — temporarily relaxing validation
3. **`ValidationMode` flags** — graduated control over which checks to skip
4. **Loading non-conformant files** — using `open_odm()` and `open_define()` with `permissive`
5. **The Load-Inspect-Fix-Validate workflow** — repairing a broken document end-to-end

## Setup

In [ ]:
import os
import tempfile

import odmlib.odm_1_3_2.model as ODM
import odmlib.define_2_1.model as DEFINE

from odmlib import (
    ValidationMode,
    permissive,
    get_mode,
    set_mode,
    open_odm,
    open_define,
    OdmlibTypeError,
    OdmlibValidationError,
)

## 1. The Problem: Strict Mode Rejects Non-Conformant Content

By default, odmlib operates in **strict mode** — every attribute is validated at assignment
time. This catches errors early, but it also means that documents with *any* violation cannot
be loaded at all.

### Invalid value sets

Certain ODM attributes restrict their values to a defined set. For example, `DataType` on
`ItemDef` must be one of the CDISC-defined types (`text`, `integer`, `float`, etc.). If a
file contains `DataType="bogus"`, strict mode raises an error immediately:

In [ ]:
try:
    item = ODM.ItemDef(OID="IT.BAD", Name="BAD", DataType="bogus", Length=8)
except (OdmlibTypeError, OdmlibValidationError) as e:
    print(f"Strict mode rejected the ItemDef:")
    print(f"  {type(e).__name__}: {e}")

### Invalid enumerations

Similarly, `Mandatory` on `ItemRef` must be `"Yes"` or `"No"`. An invalid value like
`"Maybe"` is rejected:

In [ ]:
try:
    ref = ODM.ItemRef(ItemOID="IT.TEST", Mandatory="Maybe")
except (OdmlibTypeError, OdmlibValidationError) as e:
    print(f"Strict mode rejected the ItemRef:")
    print(f"  {type(e).__name__}: {e}")

### Loading a non-conformant file

When a file on disk contains these violations, the entire document fails to load. The
`data/nonconformant_odm.xml` file has two issues:

- An `ItemDef` with `DataType="bogus"` (not a valid CDISC data type) and a missing `Name`
  attribute
- An `ItemRef` with `Mandatory="Maybe"` (should be `"Yes"` or `"No"`)

Attempting to load this file in strict mode fails at the first violation:

In [ ]:
import odmlib.odm_loader as OL
import odmlib.loader as LD

try:
    loader = LD.ODMLoader(OL.XMLODMLoader())
    loader.open_odm_document("data/nonconformant_odm.xml")
    odm = loader.root()
except Exception as e:
    print(f"Failed to load non-conformant file in strict mode:")
    print(f"  {type(e).__name__}: {e}")

This is correct behavior for production validation, but it prevents you from inspecting,
diagnosing, or repairing the file. This is a common challenge when working with Define-XML
and ODM files received from external partners, legacy systems, or tools that generate
slightly non-conformant output.

## 2. The `permissive()` Context Manager

The `permissive()` context manager temporarily switches odmlib from strict mode to permissive
mode. Inside the `with` block, validation checks are bypassed so that non-conformant values
are accepted and stored. When the block exits, strict mode is automatically restored.

Let's try creating the same `ItemDef` that failed above:

In [ ]:
with permissive():
    item = ODM.ItemDef(OID="IT.BAD", Name="BAD", DataType="bogus", Length=8)
    print(f"ItemDef created successfully in permissive mode")
    print(f"  OID:      {item.OID}")
    print(f"  Name:     {item.Name}")
    print(f"  DataType: {item.DataType}")

# Strict mode is automatically restored outside the block
print(f"\nCurrent mode: {get_mode()}")

The `DataType="bogus"` value was stored without raising an error. Outside the `with` block,
strict mode is restored, so any *new* assignments are validated normally.

Now let's load the non-conformant file. We wrap the loader calls in `permissive()` to bypass
validation during loading:

In [ ]:
with permissive():
    loader = LD.ODMLoader(OL.XMLODMLoader())
    loader.open_odm_document("data/nonconformant_odm.xml")
    odm = loader.root()

print(f"Loaded successfully in permissive mode!")
print(f"  FileOID:    {odm.FileOID}")
print(f"  Study OID:  {odm.Study[0].OID}")
print(f"  StudyName:  {odm.Study[0].GlobalVariables.StudyName}")

mdv = odm.Study[0].MetaDataVersion[0]
item_def = mdv.ItemDef[0]
item_ref = mdv.ItemGroupDef[0].ItemRef[0]

print(f"\nNon-conformant values that were loaded:")
print(f"  ItemDef.DataType:  {item_def.DataType!r} (should be a valid CDISC type)")
print(f"  ItemRef.Mandatory: {item_ref.Mandatory!r} (should be 'Yes' or 'No')")

# Asymmetry: SKIP_VALUESET/TYPE/FORMAT store the bad value on the instance, so it
# reads back normally outside the block. A missing required attribute has nothing
# stored to fall back on, so the read-time check consults the *live* validation
# mode -- reading it outside permissive() would raise OdmlibRequiredAttributeError
# even though the document was loaded permissively. Wrap reads of possibly-missing
# required attributes in `with permissive():`.
with permissive():
    print(f"  ItemDef.Name:      {item_def.Name!r} (missing required attribute)")

The file loaded despite its violations. But notice an important **asymmetry** in how the two kinds of violation behave *after* the load:

- **Invalid-but-present values** (`SKIP_VALUESET`, `SKIP_TYPE`, `SKIP_FORMAT`) — the bad value is stored on the instance during the permissive load, so it reads back normally even *outside* a `permissive()` block. That is why `ItemDef.DataType` and `ItemRef.Mandatory` above print without error.
- **Missing required attributes** (`SKIP_REQUIRED`) — there is nothing stored to return, so the missing-required check is re-evaluated on *every read* against the *live* validation mode. Reading `ItemDef.Name` outside a `permissive()` block raises `OdmlibRequiredAttributeError`, even though the document was loaded permissively. That is why the `ItemDef.Name` read above is wrapped in its own `with permissive():` block.

The practical rule: **wrap reads of possibly-missing required attributes in `permissive()`**, even when the document was originally loaded in permissive mode. Once you *assign* a value to such an attribute, it is stored on the instance and reads normally in strict mode again.

## 3. `ValidationMode` Flags: Graduated Control

Permissive mode is not all-or-nothing. The `ValidationMode` enum is a Python `Flag` that lets
you choose exactly which validation categories to skip. This is useful when you know your file
has a specific type of problem but you still want the other checks enforced.

| Flag | What it skips |
|------|---------------|
| `STRICT` | Nothing (default — all validation enforced) |
| `SKIP_REQUIRED` | Missing required attributes — returns `None` instead of raising |
| `SKIP_VALUESET` | `ValidValues` and `ExtendedValidValues` checks (e.g., `DataType`, `Mandatory` enumerations) |
| `SKIP_TYPE` | Type checks, integer/float coercion failures, and unknown attribute rejection |
| `SKIP_FORMAT` | Format validators: datetime, SAS names, email, URL, filename, regex patterns |
| `PERMISSIVE` | All of the above combined (`SKIP_REQUIRED \| SKIP_VALUESET \| SKIP_TYPE \| SKIP_FORMAT`) |

In [ ]:
# The flags and their values
for member in ValidationMode:
    print(f"  {member.name:20s} = {member.value}")

# PERMISSIVE is a composite of all four SKIP flags
print(f"\nPERMISSIVE == SKIP_REQUIRED | SKIP_VALUESET | SKIP_TYPE | SKIP_FORMAT:")
composed = (ValidationMode.SKIP_REQUIRED | ValidationMode.SKIP_VALUESET
            | ValidationMode.SKIP_TYPE | ValidationMode.SKIP_FORMAT)
print(f"  {ValidationMode.PERMISSIVE == composed}")

### SKIP_REQUIRED: Missing Required Attributes

When `SKIP_REQUIRED` is active, odmlib does not raise an error for missing required attributes
during construction. Accessing an unset required attribute returns `None` instead of raising
`OdmlibRequiredAttributeError`.

In [ ]:
# In strict mode, omitting the required Name attribute raises an error
try:
    item = ODM.ItemDef(OID="IT.TEST", DataType="text")
except Exception as e:
    print(f"Strict: {type(e).__name__}: {e}")

# With SKIP_REQUIRED, the object is created and Name returns None
with permissive(ValidationMode.SKIP_REQUIRED):
    item = ODM.ItemDef(OID="IT.TEST", DataType="text")
    print(f"\nSKIP_REQUIRED: ItemDef created without Name")
    print(f"  OID:  {item.OID}")
    print(f"  Name: {item.Name!r}  (None = not set)")

### SKIP_VALUESET: Invalid Enumeration Values

When `SKIP_VALUESET` is active, values that do not match a defined set of valid values are
accepted. This covers both `ValidValues` (dynamic lookups) and `ExtendedValidValues` (static
lists like the `DataType` or `Mandatory` enumerations).

In [ ]:
with permissive(ValidationMode.SKIP_VALUESET):
    # DataType="bogus" is not a valid CDISC data type
    item = ODM.ItemDef(OID="IT.TEST", Name="TEST", DataType="bogus", Length=8)
    print(f"SKIP_VALUESET: ItemDef.DataType = {item.DataType!r}")

    # Mandatory="Maybe" is not Yes/No
    ref = ODM.ItemRef(ItemOID="IT.TEST", Mandatory="Maybe")
    print(f"SKIP_VALUESET: ItemRef.Mandatory = {ref.Mandatory!r}")

### SKIP_TYPE: Wrong Data Types and Unknown Attributes

When `SKIP_TYPE` is active, odmlib accepts values of the wrong Python type (e.g., an integer
where a string is expected). It also allows setting unknown attributes on existing objects
via assignment, rather than rejecting them.

In [ ]:
# In strict mode, OID must be a string
try:
    study = ODM.Study(OID=12345)
except OdmlibTypeError as e:
    print(f"Strict: {e}")

# With SKIP_TYPE, the integer is stored as-is
with permissive(ValidationMode.SKIP_TYPE):
    study = ODM.Study(OID=12345)
    print(f"\nSKIP_TYPE: Study.OID = {study.OID!r} (type: {type(study.OID).__name__})")

    # Unknown attributes set via assignment are accepted instead of raising an error
    item = ODM.ItemDef(OID="IT.TEST", Name="TEST", DataType="text", Length=8)
    item.CustomAttr = "unexpected"
    print(f"SKIP_TYPE: Unknown attribute stored: CustomAttr = {item.CustomAttr!r}")

### SKIP_FORMAT: Format Validation

When `SKIP_FORMAT` is active, format validators for datetime strings, SAS names, email
addresses, URLs, and regex patterns are bypassed. This is useful when loading files that
contain malformed date/time values or non-standard SAS names.

In [ ]:
import datetime

# In strict mode, CreationDateTime must be a valid ISO 8601 datetime
try:
    odm_bad = ODM.ODM(
        FileOID="F.TEST", FileType="Snapshot",
        CreationDateTime="not-a-datetime",
        ODMVersion="1.3.2",
    )
except Exception as e:
    print(f"Strict: {type(e).__name__}: {e}")

# With SKIP_FORMAT, the malformed string is stored
with permissive(ValidationMode.SKIP_FORMAT):
    odm_bad = ODM.ODM(
        FileOID="F.TEST", FileType="Snapshot",
        CreationDateTime="not-a-datetime",
        ODMVersion="1.3.2",
    )
    print(f"\nSKIP_FORMAT: CreationDateTime = {odm_bad.CreationDateTime!r}")

### Composing Flags

Flags can be combined with the `|` operator to skip multiple validation categories while
keeping others enforced. This lets you be precise about what you relax:

In [ ]:
# Skip required and valueset checks, but still enforce type and format
selective = ValidationMode.SKIP_REQUIRED | ValidationMode.SKIP_VALUESET
print(f"Combined mode: {selective}")
print(f"  Includes SKIP_REQUIRED: {bool(selective & ValidationMode.SKIP_REQUIRED)}")
print(f"  Includes SKIP_VALUESET: {bool(selective & ValidationMode.SKIP_VALUESET)}")
print(f"  Includes SKIP_TYPE:     {bool(selective & ValidationMode.SKIP_TYPE)}")
print(f"  Includes SKIP_FORMAT:   {bool(selective & ValidationMode.SKIP_FORMAT)}")

with permissive(selective):
    # This works: missing Name (SKIP_REQUIRED) and bogus DataType (SKIP_VALUESET)
    item = ODM.ItemDef(OID="IT.TEST", DataType="bogus")
    print(f"\nCreated ItemDef with missing Name and invalid DataType")
    print(f"  Name: {item.Name!r}, DataType: {item.DataType!r}")

    # But this still fails: wrong type for OID (SKIP_TYPE is NOT active)
    try:
        bad = ODM.Study(OID=12345)
    except OdmlibTypeError:
        print(f"  Type check still enforced: OID=12345 was rejected")

## 4. Loading Non-Conformant Files with `open_odm()` and `open_define()`

The `open_odm()` and `open_define()` context managers accept a `permissive` parameter that
activates permissive mode for the duration of the load. This is the most convenient way to
load non-conformant files.

### Loading a non-conformant ODM file

Pass `permissive=True` to skip all validation, or pass a specific `ValidationMode` flag for
selective relaxation:

In [ ]:
output = os.path.join(tempfile.mkdtemp(), "inspect_odm.xml")

with open_odm("data/nonconformant_odm.xml", output_file=output, permissive=True) as odm:
    print(f"Loaded non-conformant ODM: {odm.FileOID}")
    print(f"  Study:     {odm.Study[0].OID}")
    print(f"  StudyName: {odm.Study[0].GlobalVariables.StudyName}")

    mdv = odm.Study[0].MetaDataVersion[0]
    for igd in mdv.ItemGroupDef:
        print(f"\n  ItemGroupDef: {igd.Name} ({igd.OID})")
        for ref in igd.ItemRef:
            print(f"    ItemRef: OID={ref.ItemOID}, Mandatory={ref.Mandatory!r}")

    for item in mdv.ItemDef:
        print(f"\n  ItemDef: OID={item.OID}")
        print(f"    Name:     {item.Name!r}")
        print(f"    DataType: {item.DataType!r}")

### Selective relaxation with `open_odm()`

You can pass a specific `ValidationMode` flag instead of `True` to control exactly which
checks are skipped. Only do this when you know the specific type of violation in the file:

In [ ]:
output = os.path.join(tempfile.mkdtemp(), "selective_odm.xml")
selective_mode = ValidationMode.SKIP_REQUIRED | ValidationMode.SKIP_VALUESET

with open_odm("data/nonconformant_odm.xml", output_file=output,
              permissive=selective_mode) as odm:
    mdv = odm.Study[0].MetaDataVersion[0]
    item = mdv.ItemDef[0]
    print(f"Loaded with selective flags: {selective_mode}")
    print(f"  ItemDef.Name:     {item.Name!r} (missing required — allowed by SKIP_REQUIRED)")
    print(f"  ItemDef.DataType: {item.DataType!r} (invalid value — allowed by SKIP_VALUESET)")

### Loading a non-conformant Define-XML file

The `data/nonconformant_define21.xml` file contains a Define-XML 2.1 document with:

- An `ItemGroupDef` missing the required `Repeating` attribute
- A `def:Context` attribute set to `"Other"` (not a standard submission context value)

Use `open_define()` with `permissive=True` to load it:

In [ ]:
output = os.path.join(tempfile.mkdtemp(), "inspect_define.xml")

with open_define("data/nonconformant_define21.xml", output_file=output,
                 permissive=True) as define:
    print(f"Loaded non-conformant Define-XML: {define.FileOID}")
    print(f"  Context: {define.Context!r}")
    mdv = define.Study.MetaDataVersion
    print(f"  DefineVersion: {mdv.DefineVersion}")

    for igd in mdv.ItemGroupDef:
        print(f"\n  ItemGroupDef: {igd.Name} ({igd.OID})")
        print(f"    Repeating:      {igd.Repeating!r}  (missing required attribute)")
        print(f"    SASDatasetName: {igd.SASDatasetName!r}")

    for item in mdv.ItemDef:
        print(f"\n  ItemDef: {item.Name} ({item.OID})")
        print(f"    DataType: {item.DataType!r}")

## 5. The Load-Inspect-Fix-Validate Workflow

Permissive mode enables a structured workflow for repairing non-conformant documents:

1. **Load**: open the document in permissive mode
2. **Inspect**: examine the loaded objects to find violations
3. **Fix**: correct the violations programmatically
4. **Validate**: reload the repaired document in strict mode to confirm it is conformant

### Step 1: Load and Inspect

Load the non-conformant ODM file and identify all violations:

In [ ]:
with permissive():
    loader = LD.ODMLoader(OL.XMLODMLoader())
    loader.open_odm_document("data/nonconformant_odm.xml")
    odm = loader.root()

mdv = odm.Study[0].MetaDataVersion[0]

# Inspect for violations
issues = []

for item in mdv.ItemDef:
    # Reading a possibly-missing required attribute needs permissive mode, even
    # though the document was loaded permissively -- see the asymmetry note in
    # section 2.
    with permissive():
        if item.Name is None:
            issues.append(f"ItemDef '{item.OID}': missing required Name attribute")
    if item.DataType not in ("text", "integer", "float", "date", "time",
                             "datetime", "string", "boolean", "double",
                             "hexBinary", "base64Binary", "hexFloat",
                             "base64Float", "partialDate", "partialTime",
                             "partialDatetime", "durationDatetime",
                             "intervalDatetime", "incompleteDatetime",
                             "incompleteDate", "incompleteTime", "URI"):
        issues.append(f"ItemDef '{item.OID}': invalid DataType '{item.DataType}'")

for igd in mdv.ItemGroupDef:
    for ref in igd.ItemRef:
        if ref.Mandatory not in ("Yes", "No"):
            issues.append(f"ItemRef '{ref.ItemOID}': invalid Mandatory '{ref.Mandatory}'")

print(f"Found {len(issues)} issue(s):")
for i, issue in enumerate(issues, 1):
    print(f"  {i}. {issue}")

### Step 2: Fix the Violations

Now correct each violation. Since we are outside the `permissive()` block, the new values
are validated in strict mode — guaranteeing that our fixes are themselves conformant:

In [ ]:
# Fix 1: Set the missing Name attribute
for item in mdv.ItemDef:
    # Reading the missing-required Name to detect it needs permissive mode; once
    # set on the next line, subsequent reads work in strict mode.
    with permissive():
        if item.Name is None:
            item.Name = item.OID.replace("IT.", "")  # derive Name from OID
            print(f"Fixed: ItemDef '{item.OID}' Name set to '{item.Name}'")

# Fix 2: Correct the invalid DataType
for item in mdv.ItemDef:
    if item.DataType == "bogus":
        item.DataType = "text"  # default to text
        print(f"Fixed: ItemDef '{item.OID}' DataType set to 'text'")

# Fix 3: Correct the invalid Mandatory value
for igd in mdv.ItemGroupDef:
    for ref in igd.ItemRef:
        if ref.Mandatory == "Maybe":
            ref.Mandatory = "No"  # default to No for uncertain items
            print(f"Fixed: ItemRef '{ref.ItemOID}' Mandatory set to 'No'")

### Step 3: Save the Repaired Document

Write the corrected document to a new file:

In [ ]:
repaired_file = os.path.join(tempfile.mkdtemp(), "repaired_odm.xml")
odm.write_xml(repaired_file)
print(f"Repaired document written to: {repaired_file}")

### Step 4: Validate in Strict Mode

Reload the repaired document in strict mode to confirm that all violations have been
resolved. If no exception is raised, the document is conformant:

In [ ]:
try:
    loader = LD.ODMLoader(OL.XMLODMLoader())
    loader.open_odm_document(repaired_file)
    odm_clean = loader.root()
    print("Repaired document loaded successfully in strict mode!")

    mdv = odm_clean.Study[0].MetaDataVersion[0]
    for item in mdv.ItemDef:
        print(f"  ItemDef: OID={item.OID}, Name={item.Name!r}, DataType={item.DataType!r}")
    for igd in mdv.ItemGroupDef:
        for ref in igd.ItemRef:
            print(f"  ItemRef: OID={ref.ItemOID}, Mandatory={ref.Mandatory!r}")

except Exception as e:
    print(f"Validation failed: {type(e).__name__}: {e}")

## 6. Nesting and Mode Inspection

The `permissive()` context manager supports nesting. Inner modes take effect while active,
and the outer mode is restored when the inner block exits. Use `get_mode()` to inspect the
current mode at any point:

In [ ]:
print(f"Outside: {get_mode()}")

with permissive(ValidationMode.SKIP_REQUIRED):
    print(f"Outer:   {get_mode()}")

    with permissive(ValidationMode.SKIP_REQUIRED | ValidationMode.SKIP_VALUESET):
        print(f"Inner:   {get_mode()}")

    print(f"Outer:   {get_mode()}  (restored after inner block)")

print(f"Outside: {get_mode()}  (restored after outer block)")

The mode is also restored correctly if an exception occurs inside a `permissive()` block.
This ensures that a failed permissive operation never accidentally leaves validation disabled:

In [ ]:
try:
    with permissive():
        print(f"Inside:  {get_mode()}")
        raise RuntimeError("something went wrong")
except RuntimeError:
    pass

print(f"After exception: {get_mode()}  (strict mode restored)")

## Quick Reference

### Imports

```python
from odmlib import ValidationMode, permissive, get_mode, set_mode, open_odm, open_define
```

### Common Patterns

```python
# Load with all validation skipped
with permissive():
    loader.open_odm_document("broken.xml")
    odm = loader.root()

# Load with open_odm convenience function
with open_odm("broken.xml", output_file="fixed.xml", permissive=True) as odm:
    ...

# Skip only specific validation categories
with permissive(ValidationMode.SKIP_REQUIRED | ValidationMode.SKIP_VALUESET):
    ...

# Check current mode
mode = get_mode()
```

### `ValidationMode` Flags

| Flag | Skipped Validation |
|------|--------------------|
| `STRICT` | None (default) |
| `SKIP_REQUIRED` | Missing required attributes |
| `SKIP_VALUESET` | Value set / enumeration checks |
| `SKIP_TYPE` | Type checks, unknown attributes |
| `SKIP_FORMAT` | Datetime, SAS name, email, URL format |
| `PERMISSIVE` | All of the above |

### Key Behaviors

- **Thread-safe**: uses `contextvars`, so each thread has its own mode
- **Exception-safe**: mode is always restored on block exit, even if an exception occurs
- **Nestable**: inner `permissive()` blocks can override and restore outer modes
- **Default is strict**: outside any `permissive()` block, all validation is enforced